# CBMS Split Architecture: Cloud Backend
This notebook starts the FastAPI pipeline exposed via ngrok.

In [ ]:
# 1. Install dependencies
!pip install fastapi uvicorn[standard] python-multipart opencv-python-headless insightface onnxruntime ultralytics mediapipe pyngrok nest_asyncio numpy pillow lapx

In [ ]:
# 2. Setup directory structure
import os
os.makedirs('/content/cbms/cv_pipeline/core', exist_ok=True)
os.makedirs('/content/cbms/cv_pipeline/modules', exist_ok=True)

**Paste your code contents inside the cells below.** They will be written directly to the `/content/cbms` folder structure.

In [ ]:
%%writefile /content/cbms/cv_pipeline/__init__.py
# PASTE __init__.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/core/__init__.py
# PASTE __init__.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/core/config.py
# PASTE config.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/core/database.py
# PASTE database.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/__init__.py
# PASTE __init__.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/scene_monitor.py
# PASTE scene_monitor.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/tracker.py
# PASTE tracker.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/activity_detector.py
# PASTE activity_detector.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/evidence_buffer.py
# PASTE evidence_buffer.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/face_recognizer.py
# PASTE face_recognizer.py HERE


In [ ]:
%%writefile /content/cbms/cv_pipeline/modules/rule_engine.py
# PASTE rule_engine.py HERE


### Main Application with `/ingest` endpoint
The cell below provides the `main.py` setup, including the **non-blocking `/ingest` endpoint** logic (using `asyncio.run_in_executor`) and WebSocket broadcasting. Merge this with any other routes you might have in your `main.py`.

In [ ]:
%%writefile /content/cbms/main.py
import cv2
import numpy as np
import base64
import asyncio
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse

app = FastAPI()

# TODO: Import and initialize your pipeline modules here! 
# from cv_pipeline.modules.scene_monitor import SceneMonitor
# from cv_pipeline.modules.tracker import Tracker
# from cv_pipeline.modules.activity_detector import ActivityDetector
# from cv_pipeline.modules.evidence_buffer import EvidenceBuffer
# from cv_pipeline.modules.face_recognizer import FaceRecognizer
# from cv_pipeline.modules.rule_engine import RuleEngine
# 
# scene_monitor = SceneMonitor()
# tracker = Tracker()
# activity_detector = ActivityDetector()
# evidence_buffer = EvidenceBuffer()
# face_recognizer = FaceRecognizer()
# rule_engine = RuleEngine()

def process_frame_sync(file_bytes):
    """Synchronous CV pipeline inference logic."""
    nparr = np.frombuffer(file_bytes, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if frame is None:
        return {"error": "Could not decode image", "tracks": []}

    # --- START PIPELINE EXECUTION --- 
    # 1. if not scene_monitor.should_trigger(): return ...
    # 2. tracker.update()
    # 3. activity_detector.detect()
    # 4. evidence_buffer(...)
    # 5. if complete: face_recognizer.identify_from_crops()
    # 6. alerts = rule_engine.evaluate()
    
    # For structural completion, we return a mocked standard response.
    # Real integration depends on your specific module output interfaces.
    success, encoded_image = cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    if success:
        b64_image = base64.b64encode(encoded_image.tobytes()).decode('utf-8')
    else:
        b64_image = ""
    
    # Format returning object as strictly requested
    # tracks format: [{"id": int, "bbox": [], "activity": str}]
    return {
        "annotated_frame": b64_image, 
        "tracks": [],
        "generated_alerts": []  # Optionally pass back alerts to be broadcasted
    }

@app.post("/ingest")
async def ingest_frame(file: UploadFile = File(...)):
    file_bytes = await file.read()
    # Execute heavy inference without blocking the async event loop
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(None, process_frame_sync, file_bytes)
    
    # Broadcast any fired alerts over the existing `/ws/alerts` WebSocket here 
    # (assuming you have a websocket manager instance)
    # for alert in result.get("generated_alerts", []):
    #     await websocket_manager.broadcast_alert(alert)
        
    return JSONResponse(content={
        "annotated_frame": result.get("annotated_frame", ""),
        "tracks": result.get("tracks", [])
    })

# PASTE THE REST OF YOUR MAIN.PY CODE (WEBSOCKETS ETC.) BELOW


In [ ]:
import torch
import onnxruntime as ort

device = "gpu" if torch.cuda.is_available() else "cpu"
has_cuda_ort = "CUDAExecutionProvider" in ort.get_available_providers()

print(f"PyTorch sees GPU: {torch.cuda.is_available()}")
print(f"ONNX Providers: {ort.get_available_providers()}")

if device == "gpu" and has_cuda_ort:
    print("\n✅ Successfully detected GPU hardware and CUDAExecutionProvider for ONNX. CV Pipeline should roar!")
else:
    print("\n⚠️ Running purely on CPU footprint. Fallback processing active.")

In [ ]:
import nest_asyncio
import uvicorn
import threading
import sys
from pyngrok import ngrok

sys.path.append('/content/cbms')

# Setup nested asyncio loops for colab
nest_asyncio.apply()

PORT = 8000
tunnel = ngrok.connect(PORT)
print("==========================================================")
print("🔥 BACKEND URL:", tunnel.public_url)
print("==========================================================")

def serve():
   uvicorn.run("main:app", host="0.0.0.0", port=PORT, log_level="error")

# Start FastAPI on a separate thread
server_thread = threading.Thread(target=serve, daemon=True)
server_thread.start()

In [ ]:
import time

print(f"\nServer is now active. Send requests to {tunnel.public_url}")

# Keep the Colab environment process indefinitely alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopping...")
    ngrok.kill()